# layout_algebra：CuTe layout 代数

[tensor_101](tensor_101.ipynb) 里的 layout 就是 shape:stride，v0 手写了每一维的 stride。CuTe 在 layout 之上定义了一组运算：coalesce、composition、complement、divide、product、inverse，用来做 reshape、转置、分块、按线程划分以及找最大向量宽度。FlyDSL 实现了同一套运算（`fx.composition`、`fx.logical_divide` ……）。

本实验参考 [pyhip/docs/LayoutAlgebra.ipynb](https://github.com/tingqli/pyhip/blob/main/docs/LayoutAlgebra.ipynb)，把其中的例子改用 FlyDSL 实现，再补上两点：

- 用 `right_inverse` 算出来的向量宽度，在 gfx942 ISA 里确实变成一条宽 load/store；
- v0 手写的 A layout 就是一次 `logical_divide`。

各运算的严格定义见 [CuTe Layout Algebra](https://docs.nvidia.com/cutlass/latest/media/docs/cpp/cute/02_layout_algebra.html)。

## 0. 准备

静态 layout 的运算结果就是一个新的 `!fly.layout<...>` 类型，在 host 上开一个 MLIR context 就能直接调用 `fx` 的 layout 运算，不必写 kernel。

- `text(L)`：layout 的 `shape:stride` 文本；
- `offsets(L)`：把 layout 当作 1D 函数，对下标 0, 1, 2, … 求值；
- `show(L)`：打印 layout 并列出 offset 表。rank 2 时行是 mode 0 的坐标、列是 mode 1 的坐标（嵌套的 mode 0 会标出层次坐标）；其它 rank 按 1D 下标列一行。`mn=M` 时把 offset 写成 `M×N` 列主序矩阵里的 `m,n`。

第 9 节要编译 kernel，沿用 [tensor_101](tensor_101.ipynb) 的 `isa()`/`body()`。`flyc.jit` 看到已有 MLIR context 会当作已在 jit 内部，所以 launch 前先 `host.close()`，之后再 `open_host()`。

In [1]:
import os
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"

import contextlib, glob, io, re, tempfile
from collections import Counter
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl._mlir import ir

print("flydsl", version("flydsl"), "| torch", torch.__version__,
      "| GPU", torch.cuda.get_device_properties(0).gcnArchName)

# host 上的 MLIR context：打开后，之后的 cell 可以直接调用 fx 的 layout 运算。
# flyc.jit 发现已有 context 时会当作在 jit 内部，launch kernel 前要先 host.close()。
host = contextlib.ExitStack()


def open_host():
    ctx = ir.Context()
    ctx.load_all_available_dialects()
    host.enter_context(ctx)
    host.enter_context(ir.Location.unknown())
    host.enter_context(ir.InsertionPoint(ir.Module.create().body))


open_host()


def py(x):
    return x.to_py_value()


def text(L):
    s = str(L.type)                      # !fly.layout<(4,6):(1,4)>
    return s[s.index("<") + 1:-1]


def offsets(L):
    return [py(fx.crd2idx(i, L)) for i in range(py(fx.size(L)))]


def colex(i, shape):
    # 1D 下标按列主序（mode 0 最快）拆成坐标，只用来标行号
    if isinstance(shape, int):
        return i
    crd = []
    for s in shape:
        n = py(fx.size(s))
        crd.append(colex(i % n, s))
        i //= n
    return tuple(crd)


def show(L, name="", mn=None):
    fmt = (lambda o: f"{o % mn},{o // mn}") if mn else str
    w = 6 if mn else 4
    head = f"{name} = " if name else ""
    print(f"{head}{text(L)}   size={py(fx.size(L))} cosize={py(fx.cosize(L))}")
    if L.rank != 2:
        print("            " + "".join(f"{fmt(o):>{w}}" for o in offsets(L)))
        return
    for i in range(py(fx.size(L[0]))):
        label = str(i) if L[0].is_leaf else f"{i}={colex(i, py(L[0].shape))}"
        row = [fmt(py(fx.crd2idx((i, j), L))) for j in range(py(fx.size(L[1])))]
        print(f"  {label:>9}:" + "".join(f"{v:>{w}}" for v in row))

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | GPU gfx950:sramecc+:xnack-


## 1. layout 是函数

layout 把坐标映射成 offset：$L(c_0, c_1, \dots) = \sum_i c_i \cdot s_i$。

In [2]:
show(fx.make_layout(6, 1), "a")
show(fx.make_layout((3, 5), (1, 3)), "b")
show(fx.make_layout((3, 5), (5, 1)), "c")

c = fx.make_layout((3, 5), (5, 1))
print("c(1, 1) =", py(fx.crd2idx((1, 1), c)),
      "| c(4) =", py(fx.crd2idx(4, c)),
      "| c(1) =", py(fx.crd2idx(1, c)),
      "| idx2crd(5, c) =", py(fx.idx2crd(5, c)))

a = 6:1   size=6 cosize=6
               0   1   2   3   4   5
b = (3,5):(1,3)   size=15 cosize=15
          0:   0   3   6   9  12
          1:   1   4   7  10  13
          2:   2   5   8  11  14
c = (3,5):(5,1)   size=15 cosize=15
          0:   0   1   2   3   4
          1:   5   6   7   8   9
          2:  10  11  12  13  14
c(1, 1) = 6 | c(4) = 6 | c(1) = 5 | idx2crd(5, c) = (1, 0)


- `b` 是 3×5 列主序，`c` 是 3×5 行主序，和 torch 的 shape/stride 一样。
- 坐标也可以是一个 1D 下标：先按 mode 0 最快（列主序）拆成多维坐标，再求 offset。`c(4)`：4 拆成 `(1, 1)`，offset 6；`c(1)`：1 拆成 `(1, 0)`，offset 5。所以任何 layout 都可以看成 1D 函数 $[0, \text{size}) \to \mathbb{Z}$，后面所有运算都基于这个视角。
- `idx2crd(o, L)` 是反方向，由 offset 求坐标：`c` 里 offset 5 在 `(1, 0)`。

shape 可以嵌套：

In [3]:
d = fx.make_layout(((2, 2), (2, 2)), ((1, 4), (2, 8)))
show(d, "d")
print("rank", d.rank, "depth", d.depth,
      "| d((1,0),(0,1)) =", py(fx.crd2idx(((1, 0), (0, 1)), d)),
      "| d(1, 2) =", py(fx.crd2idx((1, 2), d)))

d = ((2,2),(2,2)):((1,4),(2,8))   size=16 cosize=16
   0=(0, 0):   0   2   8  10
   1=(1, 0):   1   3   9  11
   2=(0, 1):   4   6  12  14
   3=(1, 1):   5   7  13  15
rank 2 depth 2 | d((1,0),(0,1)) = 9 | d(1, 2) = 9


- `d` 仍是 2 维（rank=2），每一维内部又分两层（depth=2）。
- 每一层都可以给完整坐标 `((1,0),(0,1))`，也可以只给一个整数 `(1, 2)`，整数按列主序展开到下一层。
- `d[0]`、`d[1]` 取出单个 mode：

In [4]:
print("d[0] =", text(d[0]), "| d[1] =", text(d[1]))

d[0] = (2,2):(1,4) | d[1] = (2,2):(2,8)


## 2. size 与 cosize

- size：定义域大小，即 shape 各维的乘积；
- cosize：最大 offset + 1，即值域需要的空间。

In [5]:
for s in [0, 1, 2]:
    show(fx.make_layout(6, s))
for s in [(1, 0), (1, 2), (1, 4)]:
    show(fx.make_layout((2, 3), s))

6:0   size=6 cosize=1
               0   0   0   0   0   0
6:1   size=6 cosize=6
               0   1   2   3   4   5
6:2   size=6 cosize=11
               0   2   4   6   8  10
(2,3):(1,0)   size=6 cosize=2
          0:   0   0   0
          1:   1   1   1
(2,3):(1,2)   size=6 cosize=6
          0:   0   2   4
          1:   1   3   5
(2,3):(1,4)   size=6 cosize=10
          0:   0   4   8
          1:   1   5   9


- cosize < size：有 offset 重复，stride 0 就是 broadcast；
- cosize = size：每个 offset 恰好出现一次，是紧凑的 layout；
- cosize > size：值域里有空洞，如 padding 过的矩阵。

## 3. coalesce：合并维度

相邻两维 $s_0{:}d_0,\ s_1{:}d_1$，若 $d_1 = s_0 d_0$，就能合成一维 $s_0 s_1 : d_0$；shape 为 1 的维直接去掉。coalesce 不改变 1D 函数，只把表示化简到最少的维数。

In [6]:
for shape, stride in [((2, 4), (1, 2)), ((2, (1, 6)), (1, (6, 2))), ((2, 4, 3), (1, 2, 16)), ((4, 2), (2, 1))]:
    L = fx.make_layout(shape, stride)
    C = fx.coalesce(L)
    print(f"{text(L):22} -> {text(C):14} 1D 函数相同: {offsets(L) == offsets(C)}")

L = fx.make_layout(((2, 4), (3, 2)), ((1, 2), (8, 24)))
print("按 mode coalesce:", text(L), "->", text(fx.coalesce(L, (1, 1))))

(2,4):(1,2)            -> 8:1            1D 函数相同: True
(2,(1,6)):(1,(6,2))    -> 12:1           1D 函数相同: True
(2,4,3):(1,2,16)       -> (8,3):(1,16)   1D 函数相同: True
(4,2):(2,1)            -> (4,2):(2,1)    1D 函数相同: True
按 mode coalesce: ((2,4),(3,2)):((1,2),(8,24)) -> (8,6):(1,8)


- `(4,2):(2,1)` 不能合并：前一维的跨度是 $4 \cdot 2 = 8 \neq 1$。
- 第二个参数 `(1, 1)` 表示保持 rank 2，只在每个 mode 内部合并。

## 4. 拼接：`fx.append`

`fx.append(A, B)` 把 B 接到 A 后面作为新的一维。新维的坐标给 offset 加上 $\{0, d, 2d, \dots\}$：

In [7]:
a = fx.make_layout(4, 1)
for d in [0, 4, 10]:
    show(fx.append(a, fx.make_layout(3, d)), f"(4:1, 3:{d})")

(4:1, 3:0) = (4,3):(1,0)   size=12 cosize=4
          0:   0   0   0
          1:   1   1   1
          2:   2   2   2
          3:   3   3   3
(4:1, 3:4) = (4,3):(1,4)   size=12 cosize=12
          0:   0   4   8
          1:   1   5   9
          2:   2   6  10
          3:   3   7  11
(4:1, 3:10) = (4,3):(1,10)   size=12 cosize=24
          0:   0  10  20
          1:   1  11  21
          2:   2  12  22
          3:   3  13  23


- `d=0`：broadcast，和已有 offset 完全重叠（cosize 4）；
- `d=4`：恰好接上，紧凑（cosize 12）；
- `d=10`：留下空洞（cosize 24）。

"找到恰好接上的那些维"就是下一节的 complement。

## 5. complement：补齐剩下的维

`R = complement(A, M)`：R 的 stride 递增，和 A 的 offset 互不重叠，而且 `(A, R)` 恰好铺满 $[0, M)$。可以理解为把 $M$ 分解成若干维之后，A 已经占了一部分维，R 是剩下的那些。

In [8]:
for shape, stride, M in [(4, 1, 12), (3, 4, 12), ((2, 2), (1, 6), 24), (4, 2, 24)]:
    A = fx.make_layout(shape, stride)
    R = fx.complement(A, M)
    full = offsets(fx.append(A, R))
    print(f"complement({text(A)}, {M}) = {text(R):12}  (A, R) 恰好覆盖 0..{M - 1}: {sorted(full) == list(range(M))}")

A = fx.make_layout(4, 2)
show(fx.append(A, fx.complement(A, 24)), "(A, A*)")

complement(4:1, 12) = 3:4           (A, R) 恰好覆盖 0..11: True
complement(3:4, 12) = 4:1           (A, R) 恰好覆盖 0..11: True
complement((2,2):(1,6), 24) = (3,2):(2,12)  (A, R) 恰好覆盖 0..23: True
complement(4:2, 24) = (2,3):(1,8)   (A, R) 恰好覆盖 0..23: True
(A, A*) = (4,(2,3)):(2,(1,8))   size=24 cosize=24
          0:   0   1   8   9  16  17
          1:   2   3  10  11  18  19
          2:   4   5  12  13  20  21
          3:   6   7  14  15  22  23


- `4:1` 占了低 2 位，补上 `3:4`；`3:4` 占了高位，补上 `4:1`。
- `4:2` 只占偶数 offset 0, 2, 4, 6，补上 `(2,3):(1,8)`：先补奇数，再补 8、16 的整块。表里每一列是 A 的一份平移，24 个 offset 各出现一次。

## 6. composition：给 layout 换视图

`R = composition(A, B)`，即 $R(c) = A(B(c))$：B 的输出被当作 A 的 1D 下标。R 的 shape 和 B 相同，所以 composition 用来给 A 换一个视图：reshape、转置、广播、采样。

先用 `6:1`（$A(k)=k$，相当于恒等映射）当 A，此时结果就是 B 本身：

In [9]:
a = fx.make_layout(6, 1)
for shape, stride in [((2, 3), (1, 2)), ((2, 3), (3, 1)), ((2, 3), (1, 0)), ((2, 2), (1, 3))]:
    B = fx.make_layout(shape, stride)
    show(fx.composition(a, B), f"6:1 o {text(B)}")

6:1 o (2,3):(1,2) = (2,3):(1,2)   size=6 cosize=6
          0:   0   2   4
          1:   1   3   5
6:1 o (2,3):(3,1) = (2,3):(3,1)   size=6 cosize=6
          0:   0   1   2
          1:   3   4   5
6:1 o (2,3):(1,0) = (2,3):(1,0)   size=6 cosize=2
          0:   0   0   0
          1:   1   1   1
6:1 o (2,2):(1,3) = (2,2):(1,3)   size=4 cosize=5
          0:   0   3
          1:   1   4


四个结果依次是：reshape 成 2×3、2×3 的转置、broadcast、采样 2×2。

换一个不是恒等的 A：6×4 列主序矩阵，列与列之间 padding 到 8（leading dimension = 8）：

In [10]:
A = fx.make_layout((6, 4), (1, 8))
show(A, "A")
for B in [fx.make_layout((4, 6), (6, 1)), fx.make_layout((2, 2), (2, 12))]:
    R = fx.composition(A, B)
    ok = all(py(fx.crd2idx(py(fx.crd2idx(i, B)), A)) == o for i, o in enumerate(offsets(R)))
    show(R, f"A o {text(B)}")
    print("    R(i) == A(B(i)):", ok)

T = fx.composition(A, fx.make_tile(fx.make_layout(3, 2), fx.make_layout(2, 1)))
show(T, "A o <3:2, 2:1>")

A = (6,4):(1,8)   size=24 cosize=30
          0:   0   8  16  24
          1:   1   9  17  25
          2:   2  10  18  26
          3:   3  11  19  27
          4:   4  12  20  28
          5:   5  13  21  29
A o (4,6):(6,1) = (4,6):(8,1)   size=24 cosize=30
          0:   0   1   2   3   4   5
          1:   8   9  10  11  12  13
          2:  16  17  18  19  20  21
          3:  24  25  26  27  28  29
    R(i) == A(B(i)): True
A o (2,2):(2,12) = (2,2):(2,16)   size=4 cosize=19
          0:   0  16
          1:   2  18
    R(i) == A(B(i)): True
A o <3:2, 2:1> = (3,2):(2,8)   size=6 cosize=13
          0:   0   8
          1:   2  10
          2:   4  12


- `A o (4,6):(6,1)`：B 把 4×6 的坐标按行主序转成 0..23，再交给 A，得到 A 的转置视图 `(4,6):(8,1)`，padding 原样保留。
- `A o (2,2):(2,12)`：取 A 的第 0、2 行和第 0、2 列。
- `fx.make_tile(L0, L1)` 是按 mode 组合：mode 0 用 `3:2` 取第 0、2、4 行，mode 1 用 `2:1` 取前两列。后面的 divide 就是这样按 mode 作用的。

## 7. divide：分块

$$A \oslash B = A \circ (B,\ B^*),\qquad B^* = \text{complement}(B, \text{size}(A))$$

结果有两个 mode：mode 0 是一个 tile 内部（按 B 的顺序取元素），mode 1 是第几个 tile。先看 1D：

In [11]:
A = fx.make_layout(12, 1)
for B in [fx.make_layout(3, 1), fx.make_layout(3, 2), fx.make_layout((3, 2), (2, 1))]:
    R = fx.logical_divide(A, B)
    Bc = fx.complement(B, 12)
    same = offsets(R) == offsets(fx.composition(A, fx.append(B, Bc)))
    show(R, f"12:1 ⊘ {text(B)}")
    print(f"    B* = {text(Bc)},  R == A o (B, B*): {same}")

12:1 ⊘ 3:1 = (3,4):(1,3)   size=12 cosize=12
          0:   0   3   6   9
          1:   1   4   7  10
          2:   2   5   8  11
    B* = 4:3,  R == A o (B, B*): True
12:1 ⊘ 3:2 = (3,(2,2)):(2,(1,6))   size=12 cosize=12
          0:   0   1   6   7
          1:   2   3   8   9
          2:   4   5  10  11
    B* = (2,2):(1,6),  R == A o (B, B*): True
12:1 ⊘ (3,2):(2,1) = ((3,2),2):((2,1),6)   size=12 cosize=12
   0=(0, 0):   0   6
   1=(1, 0):   2   8
   2=(2, 0):   4  10
   3=(0, 1):   1   7
   4=(1, 1):   3   9
   5=(2, 1):   5  11
    B* = 2:6,  R == A o (B, B*): True


每一列是一个 tile：

- `3:1`：连续 3 个元素一块；
- `3:2`：隔一个取一个，3 个一块，`B*=(2,2):(1,6)` 负责剩下的平移；
- `(3,2):(2,1)`：6 个一块，块内顺序变成 0, 2, 4, 1, 3, 5。divide 不只切块，还把 B 的顺序带进了 tile 内部。

### 2D：按 mode 分块

tiler 写成 tuple 时，每个 mode 单独做 1D divide，`None` 表示这一维不切。四种 divide 只是把结果的维排列成不同形式：

```
A              : (M, N)                    tiler: <TileM, TileN>
logical_divide : ((TileM, RestM), (TileN, RestN))
zipped_divide  : ((TileM, TileN), (RestM, RestN))
tiled_divide   : ((TileM, TileN), RestM, RestN)
flat_divide    : (TileM, TileN, RestM, RestN)
```

In [12]:
A = fx.make_layout((4, 12), (1, 4))        # 4x12 列主序
tiler = (2, fx.make_layout((3, 2), (2, 1)))
for f in [fx.logical_divide, fx.zipped_divide, fx.tiled_divide, fx.flat_divide]:
    print(f"{f.__name__:15}", text(f(A, tiler)))
print(f"{'None 不切 mode 0':15}", text(fx.logical_divide(A, (None, fx.make_layout(3, 1)))))

logical_divide  ((2,2),((3,2),2)):((1,2),((8,4),24))
zipped_divide   ((2,(3,2)),(2,2)):((1,(8,4)),(2,24))
tiled_divide    ((2,(3,2)),2,2):((1,(8,4)),2,24)
flat_divide     (2,(3,2),2,2):(1,(8,4),2,24)
None 不切 mode 0  (4,(3,4)):(1,(4,12))


最常用的是 `zipped_divide`：mode 0 是 tile 内坐标，mode 1 是 tile 坐标，`T[None, t]` 就是第 t 个 tile。把 4×12 按 2×3 切，表里写的是元素在 A 里的 `m,n`，每列是一个 tile：

In [13]:
show(fx.zipped_divide(A, (2, 3)), "zipped_divide(A, <2,3>)", mn=4)

zipped_divide(A, <2,3>) = ((2,3),(2,4)):((1,4),(2,12))   size=48 cosize=48
   0=(0, 0):   0,0   2,0   0,3   2,3   0,6   2,6   0,9   2,9
   1=(1, 0):   1,0   3,0   1,3   3,3   1,6   3,6   1,9   3,9
   2=(0, 1):   0,1   2,1   0,4   2,4   0,7   2,7  0,10  2,10
   3=(1, 1):   1,1   3,1   1,4   3,4   1,7   3,7  1,10  3,10
   4=(0, 2):   0,2   2,2   0,5   2,5   0,8   2,8  0,11  2,11
   5=(1, 2):   1,2   3,2   1,5   3,5   1,8   3,8  1,11  3,11


tiler 也可以是一个 layout，这时它作用在 A 的 1D 下标上，而不是按 mode。`(2,2):(1,4)` 在 4×4 列主序里正好是一个 2×2 块，结果和按 mode 的 `zipped_divide(A, <2,2>)` 相同：

In [14]:
A = fx.make_layout((4, 4), (1, 4))
print("logical_divide(A, (2,2):(1,4)) =", text(fx.logical_divide(A, fx.make_layout((2, 2), (1, 4)))))
print("zipped_divide(A, <2,2>)        =", text(fx.zipped_divide(A, (2, 2))))
show(fx.logical_divide(A, fx.make_layout((2, 2), (2, 8))), "logical_divide(A, (2,2):(2,8))", mn=4)

logical_divide(A, (2,2):(1,4)) = ((2,2),(2,2)):((1,4),(2,8))
zipped_divide(A, <2,2>)        = ((2,2),(2,2)):((1,4),(2,8))
logical_divide(A, (2,2):(2,8)) = ((2,2),(2,2)):((2,8),(1,4))   size=16 cosize=16
   0=(0, 0):   0,0   1,0   0,1   1,1
   1=(1, 0):   2,0   3,0   2,1   3,1
   2=(0, 1):   0,2   1,2   0,3   1,3
   3=(1, 1):   2,2   3,2   2,3   3,3


`(2,2):(2,8)` 选出的是隔行隔列的 4 个元素，需要自己算好 A 的 1D 下标才能写对，不如按 mode 的 tiler 直观。实际中 2D 分块基本都用 tuple tiler。

## 8. product：重复

$$A \otimes B = (A,\ A^* \circ B),\qquad A^* = \text{complement}(A, \text{size}(A) \cdot \text{cosize}(B))$$

mode 0 是 A 本身，mode 1 把 A 平移复制 size(B) 份，复制的顺序由 B 决定：

In [15]:
show(fx.logical_product(fx.make_layout(3, 1), fx.make_layout(2, 1)), "3:1 ⊗ 2:1")
show(fx.logical_product(fx.make_layout(3, 1), fx.make_layout((2, 2), (2, 1))), "3:1 ⊗ (2,2):(2,1)")

3:1 ⊗ 2:1 = (3,2):(1,3)   size=6 cosize=6
          0:   0   3
          1:   1   4
          2:   2   5
3:1 ⊗ (2,2):(2,1) = (3,(2,2)):(1,(6,3))   size=12 cosize=12
          0:   0   6   3   9
          1:   1   7   4  10
          2:   2   8   5  11


第二个例子 B=`(2,2):(2,1)` 是 2×2 行主序，A 的 4 份副本（起点 0, 3, 6, 9）按 0, 6, 3, 9 的顺序排进 mode 1。

2D 常用 `blocked_product` 和 `raked_product`：都把 2×2 的 A 按 2×3 的 B 铺成 4×6，区别在 A 的元素是聚在一起还是分散开。表里 offset 0..3 是 A 的第一份：

In [16]:
A = fx.make_layout((2, 2), (1, 2))
B = fx.make_layout((2, 3), (1, 2))
show(fx.blocked_product(A, B), "blocked_product(A, B)")
show(fx.raked_product(A, B), "raked_product(A, B)")

blocked_product(A, B) = ((2,2),(2,3)):((1,4),(2,8))   size=24 cosize=24
   0=(0, 0):   0   2   8  10  16  18
   1=(1, 0):   1   3   9  11  17  19
   2=(0, 1):   4   6  12  14  20  22
   3=(1, 1):   5   7  13  15  21  23
raked_product(A, B) = ((2,2),(3,2)):((4,1),(8,2))   size=24 cosize=24
   0=(0, 0):   0   8  16   2  10  18
   1=(1, 0):   4  12  20   6  14  22
   2=(0, 1):   1   9  17   3  11  19
   3=(1, 1):   5  13  21   7  15  23


- blocked：0..3 在左上角 2×2 块里，每份 A 是一个连续的块；
- raked：0..3 在第 0、2 行和第 0、3 列，每份 A 均匀撒开（cyclic 分布）。

product 同样有 logical/zipped/tiled/flat 四种形式，维的排列方式和 divide 对应。

## 9. right_inverse 与向量宽度

把数据从 layout A 拷到 layout B（逻辑 shape 相同，排列不同），一条指令最多能拷几个元素？需要找一组逻辑坐标，它们在 A 和 B 里的 offset 都连续。

- `right_inverse(A)`：$A \circ A^{-1} = \text{id}$，把 A 的 offset 映射回 1D 下标；
- $B \circ A^{-1}$：A 的 offset → B 的 offset。从 0 开始满足 $B \circ A^{-1}(k) = k$ 的最长前缀长度 K，就是两边都连续的元素个数；
- 这 K 个元素的逻辑坐标是 $A^{-1} \circ K{:}1$，用它做 tiler 来 `zipped_divide`，tile 内在 A、B 里都是连续的。

例子来自 Cris Cecka 的论文 *CuTe Layout Representation and Algebra* 3.4.2 节：

In [17]:
A = fx.make_layout(((2, 2), (2, 2)), ((8, 2), (4, 1)))
B = fx.make_layout(((2, 2), (2, 2)), ((4, 2), (8, 1)))
show(A, "A")
show(B, "B")

Ainv = fx.right_inverse(A)
show(Ainv, "A⁻¹")
show(fx.composition(A, Ainv), "A o A⁻¹")
BA = offsets(fx.composition(B, Ainv))
show(fx.composition(B, Ainv), "B o A⁻¹")
K = next((k for k, o in enumerate(BA) if o != k), len(BA))
print("K =", K)

atom = fx.composition(Ainv, fx.make_layout(K, 1))
show(atom, "atom = A⁻¹ o K:1")
for name, L in [("A", A), ("B", B)]:
    Z = fx.zipped_divide(L, atom)
    print(f"zipped_divide({name}, atom) = {text(Z):34} tile 内: {text(fx.coalesce(Z[0]))}")

A = ((2,2),(2,2)):((8,2),(4,1))   size=16 cosize=16
   0=(0, 0):   0   4   1   5
   1=(1, 0):   8  12   9  13
   2=(0, 1):   2   6   3   7
   3=(1, 1):  10  14  11  15
B = ((2,2),(2,2)):((4,2),(8,1))   size=16 cosize=16
   0=(0, 0):   0   8   1   9
   1=(1, 0):   4  12   5  13
   2=(0, 1):   2  10   3  11
   3=(1, 1):   6  14   7  15
A⁻¹ = (2,4,2):(8,2,1)   size=16 cosize=16
               0   8   2  10   4  12   6  14   1   9   3  11   5  13   7  15
A o A⁻¹ = (2,4,2):(1,2,8)   size=16 cosize=16
               0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15
B o A⁻¹ = (2,(2,2),2):(1,(2,8),4)   size=16 cosize=16
               0   1   2   3   8   9  10  11   4   5   6   7  12  13  14  15
K = 4
atom = A⁻¹ o K:1 = (2,2):(8,2)   size=4 cosize=11
          0:   0   2
          1:   8  10
zipped_divide(A, atom) = ((2,2),(2,2)):((1,2),(8,4))        tile 内: 4:1
zipped_divide(B, atom) = ((2,2),(2,2)):((1,2),(4,8))        tile 内: 4:1


- A、B 各自都有连续的元素，但位置不同：A 的 offset 0..3 在 `atom` 选出的 4 个坐标上，B 的同样这 4 个坐标也恰好是 0..3，第 5 个就不是了，所以 K=4。
- 切完之后 tile 内在 A、B 里都 coalesce 成 `4:1`。

### 在 kernel 里验证

同样的运算可以直接写在 kernel 里。下面每个 lane 拷一个 tile：`D[None, tid].store(S[None, tid].load())`，S、D 分别按 A、B 解释 16 个 fp32。对比两种 tiler：

- `atom`：上面算出来的 $A^{-1} \circ 4{:}1$；
- `4:1`：直接取前 4 个逻辑坐标，在 A 里是 offset 0, 8, 2, 10，不连续。

In [18]:
ISA_ARCH = "gfx942"
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)


def isa(launcher, *args):
    dump_dir = tempfile.mkdtemp(prefix="layout_algebra_")
    env = dict(ARCH=ISA_ARCH, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=dump_dir)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            flyc.compile(launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    path, = glob.glob(f"{dump_dir}/**/*final_isa.s", recursive=True)
    return open(path).read()


def body(text):
    lines, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and (not line.startswith(".") or line.startswith(".LBB")):
            lines.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines


A_spec = (((2, 2), (2, 2)), ((8, 2), (4, 1)))
B_spec = (((2, 2), (2, 2)), ((4, 2), (8, 1)))


def make_copy(use_inverse):
    # kernel 有自己的 MLIR context，layout 要在 kernel 里重新构造
    @flyc.kernel(known_block_size=(4, 1, 1))
    def copy_kernel(S: fx.Tensor, D: fx.Tensor):
        tid = fx.thread_idx.x
        A = fx.make_layout(*A_spec)
        B = fx.make_layout(*B_spec)
        # kernel 里的 if 语句会被 AST 改写，分支里新定义的变量拿不到，这里用条件表达式
        atom = fx.composition(fx.right_inverse(A), fx.make_layout(4, 1)) if use_inverse else fx.make_layout(4, 1)
        S = fx.zipped_divide(fx.make_view(fx.get_iter(S), A), atom)
        D = fx.zipped_divide(fx.make_view(fx.get_iter(D), B), atom)
        print(f"  use_inverse={use_inverse}: S[None, tid] = {S[None, tid].type}")
        D[None, tid].store(S[None, tid].load())

    @flyc.jit
    def launch(S: fx.Tensor, D: fx.Tensor, stream: fx.Stream):
        copy_kernel(S, D).launch(grid=(1, 1, 1), block=(4, 1, 1), stream=stream)

    return launch


S = torch.randn(16, device="cuda")
D = torch.zeros(16, device="cuda")
expect = torch.empty_like(S)
expect[offsets(fx.make_layout(*B_spec))] = S[offsets(fx.make_layout(*A_spec))]   # D[B(c)] = S[A(c)]

host.close()
try:
    for use_inverse in [True, False]:
        D.zero_()
        make_copy(use_inverse)(S, D, stream)
        torch.cuda.synchronize()
        ops = Counter(l.split()[0] for l in body(isa(make_copy(use_inverse), S, D, stream)) if l.startswith("global_"))
        print(f"    GPU 结果正确: {torch.equal(D, expect)} | gfx942 访存指令: {dict(ops)}")
finally:
    open_host()

  use_inverse=True: S[None, tid] = !fly.memref<f32, global, (2,2):(1,2)>
    GPU 结果正确: True | gfx942 访存指令: {'global_load_dwordx4': 1, 'global_store_dwordx4': 1}
  use_inverse=False: S[None, tid] = !fly.memref<f32, global, (2,2):(8,2)>
    GPU 结果正确: True | gfx942 访存指令: {'global_load_dword': 4, 'global_store_dword': 4}


- `atom` 切出的子 tensor 是 `(2,2):(1,2)`，连续 4 个 fp32，读写各一条 `dwordx4`；
- `4:1` 切出的子 tensor 在 S 里是 `(2,2):(8,2)`，读拆成 4 条 `global_load_dword`；它在 D 里的 offset 是 0, 4, 2, 6，写也拆成 4 条。
- 两种写法结果都对，layout 运算本身不产生指令，只决定了子 tensor 的 layout，进而决定访存宽度（和 tensor_101 第 5 节的结论一致）。

## 10. v0 的 A layout 就是一次 divide

v0 把行主序的 `A[M, K]` 写成 `make_ordered_layout((8, K//8, 16, BM//16, M//BM), 0)`。按 `(k, m)` 看，A 是 `(K, M):(1, K)`；对 k 按 8 分块，对 m 按 BM 分块、块内再拆成 `(16, BM//16)`：

In [19]:
M, K, BM = 64, 64, 32


def flat(t):
    return sum((flat(x) for x in t), ()) if isinstance(t, tuple) else (t,)


A = fx.make_layout((K, M), (1, K))
T = fx.logical_divide(A, (8, fx.make_layout((16, BM // 16), (1, 16))))
v0 = fx.make_ordered_layout((8, K // 8, 16, BM // 16, M // BM), 0)
print("logical_divide:", text(T))
print("v0            :", text(v0))
print("展平后相同:", (flat(py(T.shape)), flat(py(T.stride))) == (py(v0.shape), py(v0.stride)),
      "| 1D 函数相同:", offsets(T) == offsets(v0))

logical_divide: ((8,8),((16,2),2)):((1,8),((64,1024),2048))
v0            : (8,8,16,2,2):(1,8,64,1024,2048)
展平后相同: True | 1D 函数相同: True


`logical_divide` 的结果展平后就是 v0 手写的 5 维 layout：`(8, K/8)` 是 k 的 (块内, 第几块)，`((16, BM/16), M/BM)` 是 m 的 ((块内行, 块内第几组 16 行), 第几个 BM 块)。v0 里 `A[None, None, None, None, bid_m]` 和 `A[None, kk, row, i]` 的每个坐标都能在这里找到对应。

## 总结

| 运算 | FlyDSL | 含义 | 例子 |
|---|---|---|---|
| 求值 | `fx.crd2idx(c, L)` / `fx.idx2crd(o, L)` | 坐标 → offset / offset → 坐标；整数坐标按列主序展开 | `(3,5):(5,1)`：4 → (1,1) → 6 |
| size / cosize | `fx.size` / `fx.cosize` | 定义域大小 / 最大 offset+1 | `6:2`：6 / 11 |
| coalesce | `fx.coalesce(L[, pattern])` | 合并 $d_1 = s_0 d_0$ 的相邻维，1D 函数不变 | `(2,4):(1,2)` → `8:1` |
| 拼接 | `fx.append(A, B)` | B 作为新的一维 | `(4:1, 3:4)` → `(4,3):(1,4)` |
| complement | `fx.complement(A, M)` | 和 A 一起恰好铺满 $[0, M)$ 的剩余维 | `4:2`, 24 → `(2,3):(1,8)` |
| composition | `fx.composition(A, B)`，按 mode 用 `fx.make_tile` | $A(B(c))$：reshape、转置、广播、采样 | `(6,4):(1,8) o (4,6):(6,1)` → `(4,6):(8,1)` |
| divide | `fx.logical/zipped/tiled/flat_divide` | $A \circ (B, B^*)$：(tile 内, 第几个 tile)，tile 内顺序跟随 B | `12:1 ⊘ 3:2` → `(3,(2,2)):(2,(1,6))` |
| product | `fx.logical/blocked/raked_product` 等 | $(A, A^* \circ B)$：把 A 复制 size(B) 份 | `3:1 ⊗ 2:1` → `(3,2):(1,3)` |
| right_inverse | `fx.right_inverse(A)` | offset → 下标；$B \circ A^{-1}$ 的恒等前缀 = 最大向量宽度 | 第 9 节 K=4，ISA 中为 `dwordx4` |

- 静态 layout 的运算全部在类型上完成，host 上和 kernel 里写法相同，不生成任何指令；
- 它们只决定子 tensor 的 layout，进而决定 `.load()/.store()` 的指令宽度；
- v0 手写的 `make_ordered_layout((8, K//8, 16, BM//16, M//BM), 0)` 等于 `logical_divide((K,M):(1,K), <8, (16,BM/16):(1,16)>)` 展平。